# requires-grad-propagation — worked example 1: The Three-Gate AND Rule for requires_grad Propagation

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `requires-grad-propagation`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

An operation's output has `requires_grad=True` only when all three conditions hold simultaneously: (1) global gradient tracking is enabled (not inside `torch.no_grad()`), (2) the operation is differentiable, and (3) at least one input tensor has `requires_grad=True`. This is a logical AND over three gates. If any gate is False, the output is not tracked — even if one input needs a gradient.

## Worked solution

**Step 1 — frame the rule.** The output's `requires_grad` = `grad_tracking_enabled AND is_differentiable AND any(input.requires_grad)`. All three must be True.

**Step 2 — implement with proper filtering.** When scanning inputs for the `any(...)` gate, only check tensors — not Python scalars, shape tuples, or other non-tensor args. Use `isinstance(a, t.Tensor) and a.requires_grad`.

**Step 3 — demonstrate each gate.** Show three examples where one gate is False while the others are True, proving each gate independently controls the output.

**Step 4 — demonstrate all gates True.** Only when all three hold does the output get `requires_grad=True`.

**Step 5 — relate to real PyTorch behavior.** Verify the rule matches `torch.add` inside and outside `no_grad`.

In [ ]:
import torch as t

def propagate_requires_grad(
    args: tuple,
    is_differentiable: bool,
    grad_tracking_enabled: bool,
) -> bool:
    """Three-gate AND rule for requires_grad propagation."""
    any_input_tracked = any(
        isinstance(a, t.Tensor) and a.requires_grad for a in args
    )
    return grad_tracking_enabled and is_differentiable and any_input_tracked

# --- exercise and print ---
leaf = t.tensor([1.0], requires_grad=True)
const = t.tensor([2.0])  # no grad
scalar = 3.0             # not a tensor

cases = [
    ('all gates True',         (leaf, const, scalar),  True,  True),
    ('tracking disabled',      (leaf, const),           True,  False),
    ('op not differentiable',  (leaf, const),           False, True),
    ('no input needs grad',    (const, scalar),         True,  True),
    ('mixed tensor/non-tensor',(leaf, scalar, 42),      True,  True),
]

for name, args, is_diff, tracking in cases:
    result = propagate_requires_grad(args, is_diff, tracking)
    print(f'{name:35s}: {result}')